In [ ]:
# ============================================================
# EXPERIMENT 14A: ANN FROM SCRATCH
# Synthetic Non-Linear Dataset - Binary Classification
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 1-3: Set Seed, Generate Non-Linear Dataset, and Create DataFrame
np.random.seed(42)
n_samples = 1000
X = np.random.randn(n_samples, 2)
y = (X[:, 0]**2 + X[:, 1]**2 > 1.0).astype(int)   # Class 1: x1^2 + x2^2 > 1
print(f"Dataset Shape:\n{X.shape}\n\nTarget Shape:\n{y.shape}")

df = pd.DataFrame({"Feature_1": X[:, 0], "Feature_2": X[:, 1], "Target": y})
print(f"\nFirst Five Records:\n{df.head()}")

# Step 4: Train-Test Split From Scratch
indices = np.random.permutation(n_samples)
split = int(0.80 * n_samples)
X_train, y_train = X[indices[:split]], y[indices[:split]]
X_test, y_test = X[indices[split:]], y[indices[split:]]
print(f"\nTraining Samples: {len(X_train)}\nTesting Samples: {len(X_test)}")

# Step 5: Activation Functions
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -50, 50)))
def sigmoid_derivative(output): return output * (1 - output)
def relu(x): return np.maximum(0, x)
def relu_derivative(x): return (x > 0).astype(float)

# Step 6-8: Architecture, Weight Initialization, and Display
input_size, hidden_size_1, hidden_size_2, output_size = 2, 16, 8, 1
W1 = np.random.randn(input_size, hidden_size_1) * np.sqrt(2 / input_size)
W2 = np.random.randn(hidden_size_1, hidden_size_2) * np.sqrt(2 / hidden_size_1)
W3 = np.random.randn(hidden_size_2, output_size) * np.sqrt(2 / hidden_size_2)
b1, b2, b3 = np.zeros((1, hidden_size_1)), np.zeros((1, hidden_size_2)), np.zeros((1, output_size))

print("\n===================================\nANN ARCHITECTURE\n===================================")
print(f"Input Layer: {input_size} neurons\nHidden Layer 1: {hidden_size_1} neurons\nHidden Layer 2: {hidden_size_2} neurons\nOutput Layer: {output_size} neuron")

# Step 9 & 10: Forward Propagation and Binary Cross-Entropy Loss
def forward_propagation(X_in):
    Z1 = np.dot(X_in, W1) + b1; A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2;   A2 = relu(Z2)
    return Z1, A1, Z2, A2, sigmoid(np.dot(A2, W3) + b3)

def binary_cross_entropy(y_true, y_pred):
    y_pred, y_true = np.clip(y_pred, 1e-7, 1 - 1e-7), y_true.reshape(-1, 1)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Step 11: Backpropagation
def backward_propagation(X_in, y_in, Z1, A1, Z2, A2, A3):
    m, y_in = X_in.shape[0], y_in.reshape(-1, 1)
    dZ3 = A3 - y_in
    dW3, db3 = np.dot(A2.T, dZ3) / m, np.mean(dZ3, axis=0, keepdims=True)
    dZ2 = np.dot(dZ3, W3.T) * relu_derivative(Z2)
    dW2, db2 = np.dot(A1.T, dZ2) / m, np.mean(dZ2, axis=0, keepdims=True)
    dZ1 = np.dot(dZ2, W2.T) * relu_derivative(Z1)
    dW1, db1 = np.dot(X_in.T, dZ1) / m, np.mean(dZ1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2, dW3, db3

# Step 12 & 13: Training Parameters and Training Loop
epochs, learning_rate = 1000, 0.01
loss_history, accuracy_history = [], []

for epoch in range(epochs):
    Z1, A1, Z2, A2, A3 = forward_propagation(X_train)
    loss = binary_cross_entropy(y_train, A3)
    accuracy = np.mean((A3 >= 0.5).astype(int).flatten() == y_train)
    dW1, db1, dW2, db2, dW3, db3 = backward_propagation(X_train, y_train, Z1, A1, Z2, A2, A3)
    W1 -= learning_rate * dW1; b1 -= learning_rate * db1
    W2 -= learning_rate * dW2; b2 -= learning_rate * db2
    W3 -= learning_rate * dW3; b3 -= learning_rate * db3
    loss_history.append(loss), accuracy_history.append(accuracy)
    if epoch == 0 or (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:04d} | Loss: {loss:.4f} | Accuracy: {accuracy:.4f}")

# Step 14-16: Test ANN, Display Performance, and Actual vs Predicted
A3_test = forward_propagation(X_test)[-1]
test_predictions = (A3_test >= 0.5).astype(int).flatten()
test_accuracy = np.mean(test_predictions == y_test)

print("\n===================================\nANN FROM SCRATCH PERFORMANCE\n===================================")
print(f"Test Accuracy: {round(test_accuracy, 4)}\nTest Accuracy (%): {round(test_accuracy * 100, 2)}")

results = pd.DataFrame({"Actual": y_test, "Predicted": test_predictions, "Probability": A3_test.flatten()})
print(f"\nActual vs Predicted:\n{results.head(20)}")

# Step 17: Plot Training Loss
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel("Epoch"), plt.ylabel("Binary Cross-Entropy Loss"), plt.title("ANN From Scratch - Training Loss")
plt.grid(True), plt.tight_layout(), plt.show()

# Step 18: Plot Training Accuracy
plt.figure(figsize=(8, 5))
plt.plot(accuracy_history)
plt.xlabel("Epoch"), plt.ylabel("Accuracy"), plt.title("ANN From Scratch - Training Accuracy")
plt.grid(True), plt.tight_layout(), plt.show()

# Step 19: Actual vs Predicted Plot
n = min(50, len(y_test))
idx = np.arange(n)
plt.figure(figsize=(10, 5))
plt.scatter(idx, y_test[:n], label="Actual", marker="o")
plt.scatter(idx, test_predictions[:n], label="Predicted", marker="x")
plt.xlabel("Test Sample Number"), plt.ylabel("Class"), plt.title("ANN From Scratch: Actual vs Predicted")
plt.yticks([0, 1], ["Class 0", "Class 1"]), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 20: Visualize Synthetic Dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Class 1")
plt.xlabel("Feature 1"), plt.ylabel("Feature 2"), plt.title("Synthetic Non-Linear Dataset")
plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 21 & 22: Predict and Display a New Sample
new_sample = np.array([[1.2, 0.8]])
new_probability = forward_propagation(new_sample)[-1]
new_prediction = int(new_probability[0, 0] >= 0.5)

print("\n===================================\nNEW SAMPLE PREDICTION\n===================================")
print(f"Feature 1: {new_sample[0, 0]}\nFeature 2: {new_sample[0, 1]}\nPredicted Probability: {round(new_probability[0, 0], 4)}\nPredicted Class: {new_prediction}")

# Step 23: Save Results
results.to_csv("ANN_From_Scratch_Results.csv", index=False)
print("\nResults saved as:\nANN_From_Scratch_Results.csv")

In [ ]:
# ============================================================
# EXPERIMENT 14B: ANN CLASSIFICATION (Offline-Safe Version)
# Using scikit-learn MLPClassifier on Breast Cancer Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1-5: Load Dataset, Create DataFrame, and Display Info
data = load_breast_cancer()
X, y = data.data, data.target
feature_names, target_names = data.feature_names, data.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = [target_names[i] for i in y]

print(f"Dataset Shape:\n{X.shape}\n\nTarget Classes:\n{target_names}")
print(f"\nFirst Five Records:\n{df.head()}")
print("\nDataset Information:")
df.info()
print(f"\nDataset Description:\n{df.describe()}")
print(f"\n===================================\nCLASS DISTRIBUTION\n===================================\n{df['Class'].value_counts()}")

# Step 6-7: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 8: Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print(f"\nScaled Training Shape:\n{X_train_scaled.shape}\n\nScaled Testing Shape:\n{X_test_scaled.shape}")

# Step 9-10: Create ANN Model (equivalent to Keras Sequential)
model = MLPClassifier(
    hidden_layer_sizes=(32, 16),   # two hidden layers, 32 and 16 neurons
    activation='relu',
    solver='adam',
    alpha=1e-5,                    # L2 regularization (similar to Dropout's effect)
    max_iter=1000,
    random_state=42
)
print("\n===================================\nANN MODEL CONFIGURATION\n===================================")
print(model)

# Step 11-12: Train ANN
model.fit(X_train_scaled, y_train)

# Step 13: Evaluate Model and Display Performance
train_accuracy = model.score(X_train_scaled, y_train)
test_accuracy  = model.score(X_test_scaled, y_test)
print("\n===================================\nANN PERFORMANCE\n===================================")
print(f"Training Accuracy: {round(train_accuracy, 4)}")
print(f"Test Accuracy    : {round(test_accuracy, 4)}")
print(f"Test Accuracy (%): {round(test_accuracy * 100, 2)}")

# Step 14-16: Predictions, Actual vs Predicted, and Accuracy
probabilities = model.predict_proba(X_test_scaled)[:, 1]
predictions   = model.predict(X_test_scaled)
results = pd.DataFrame({"Actual": y_test, "Predicted": predictions, "Probability": probabilities})
print(f"\n===================================\nACTUAL VS PREDICTED\n===================================\n{results.head(20)}")

accuracy = accuracy_score(y_test, predictions)
print(f"\nAccuracy: {round(accuracy, 4)}\nAccuracy (%): {round(accuracy * 100, 2)}")

# Steps 17-18: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, predictions)
print(f"\n===================================\nCONFUSION MATRIX\n===================================\n{cm}")
print(f"\n===================================\nCLASSIFICATION REPORT\n===================================\n{classification_report(y_test, predictions, target_names=target_names)}")

# Step 19: Plot Training Loss (MLPClassifier stores loss_curve_)
plt.figure(figsize=(8, 5))
plt.plot(model.loss_curve_, marker="o", label="Training Loss")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("ANN Training Loss Curve")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# Step 20: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("ANN Confusion Matrix")
plt.xlabel("Predicted Class"); plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names); plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.colorbar(); plt.tight_layout(); plt.show()

# Step 21: Actual vs Predicted Plot
n = min(50, len(y_test))
idx = np.arange(n)
plt.figure(figsize=(10, 5))
plt.scatter(idx, y_test[:n], label="Actual", marker="o")
plt.scatter(idx, predictions[:n], label="Predicted", marker="x")
plt.xlabel("Test Sample Number"); plt.ylabel("Class")
plt.title("ANN: Actual vs Predicted")
plt.yticks([0, 1], target_names)
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# Step 22-23: Predict and Display a New Patient
new_patient_scaled = scaler.transform(X_test[0].reshape(1, -1))
new_probability    = model.predict_proba(new_patient_scaled)[0][1]
new_prediction     = int(new_probability >= 0.5)
print("\n===================================\nNEW PATIENT PREDICTION\n===================================")
print(f"Predicted Probability: {round(new_probability, 4)}")
print(f"Predicted Class: {target_names[new_prediction]}")
print(f"Actual Class: {target_names[y_test[0]]}")

# Step 24: Save Results
results.to_csv("Breast_Cancer_ANN_Results.csv", index=False)
print("\nResults saved as: Breast_Cancer_ANN_Results.csv")